## Assignment-16 Recommendation System

In [1]:
import pandas as pd
import numpy as np


In [3]:
data = pd.read_csv(r'Data\anime.csv')
data.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None


In [5]:
data.shape

(12294, 7)

In [6]:
print(data.isnull().sum())

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [9]:
# Fill missing values
data['genre'] = data['genre'].fillna('')
data['type'] = data['type'].fillna(data['type'].mode()[0])
data['rating'] = data['rating'].fillna(data['rating'].mean())
data['episodes'] = data['episodes'].replace('Unknown', np.nan)
data['episodes'] = pd.to_numeric(data['episodes'], errors='coerce')
data['episodes'] = data['episodes'].fillna(data['episodes'].median())

In [10]:
print(data.isnull().sum())

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


In [12]:
features = data[['name','genre','rating','episodes']]

##### Feature Engineering

Convert Genre → Numerical (Text Vectorization)

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
cv = CountVectorizer(tokenizer=lambda x: x.split(','))

genre_matrix = cv.fit_transform(features['genre'])

C:\Users\javee\AppData\Local\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Normalize Numerical Features

In [21]:
from sklearn.preprocessing import MinMaxScaler

In [22]:
scaler = MinMaxScaler()

numeric_features = scaler.fit_transform(features[['rating', 'episodes']])

Combine Features

In [23]:
from scipy.sparse import hstack

combined_features = hstack([genre_matrix, numeric_features])

Cosine Similarity

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
cosine_sim = cosine_similarity(combined_features)
cosine_sim 

array([[1.        , 0.13660032, 0.13644327, ..., 0.15085848, 0.15492584,
        0.1737458 ],
       [0.13660032, 1.        , 0.36143579, ..., 0.11709589, 0.12023337,
        0.13483898],
       [0.13644327, 0.36143579, 1.        , ..., 0.11695735, 0.12009513,
        0.13468395],
       ...,
       [0.15085848, 0.11709589, 0.11695735, ..., 1.        , 0.99994463,
        0.99824866],
       [0.15492584, 0.12023337, 0.12009513, ..., 0.99994463, 1.        ,
        0.99881138],
       [0.1737458 , 0.13483898, 0.13468395, ..., 0.99824866, 0.99881138,
        1.        ]], shape=(12294, 12294))

Recommendation Function

In [28]:
def recommend_anime(anime_name, top_n=5):
    # Find index
    idx = data[data['name'] == anime_name].index[0]

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remove itself
    sim_scores = sim_scores[1:top_n+1]

    # Get indices
    anime_indices = [i[0] for i in sim_scores]

    return data['name'].iloc[anime_indices]

In [31]:
recommend_anime('Naruto')

615                                    Naruto: Shippuuden
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
Name: name, dtype: object

Add Threshold (Optional Improvement)

In [33]:
def recommend_with_threshold(anime_name, threshold=0.5):
    idx = data[data['name'] == anime_name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    filtered = [i for i in sim_scores if i[1] > threshold]

    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)[1:]

    anime_indices = [i[0] for i in filtered]

    return data['name'].iloc[anime_indices]

In [34]:
print(recommend_with_threshold('Naruto'))

615                                    Naruto: Shippuuden
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
                              ...                        
3096                                       Dash! Yonkurou
3066                              Kaitou Joker 3rd Season
3153                           YAT Anshin! Uchuu Ryokou 2
3022                         Guilty Crown: Lost Christmas
3144                           Overlord: Ple Ple Pleiades
Name: name, Length: 977, dtype: object


### Interview Question:


#### 1.Difference: User-based vs Item-based Collaborative Filtering

User-Based: Finds similar users

Recommends items liked by similar users

Example:

Users like you watched Naruto → recommend Naruto

Item-Based:

Finds similar items

Recommends similar items to what user liked

Example:

Naruto → recommend Bleach, One Piece

What is Collaborative Filtering?how does it work?
It is a technique that:

Uses user behavior (ratings, preferences) Recommends items based on similarity Types: User-based Item-based

how does it work:

Step-by-Step

Look at anime details
Each anime has:

Genre (Action, Comedy, etc.) Rating Episodes



#### 2.Convert everything into numbers

Computer cannot understand words, so:

“Action” → number “Comedy” → number

Compare anime
System compares one anime with all others.

Example:

Naruto vs Bleach Naruto vs One Piece

Check similarity
It checks:

Same genre? Similar rating? Similar length?

5.Give recommendations

It shows the most similar anime

Click to add a cell.